# 06 — Error Analysis and Grad-CAM

**Project:** IT22638168 — MaternaLink FER track
**Pipeline position:** step 6 of 8 · see `../README.md`

## Purpose

- Categorise the failure modes.
- Grad-CAM on correct and incorrect predictions -> plots/.
- Analyse pose effects — the literature flags extreme pose as a deployment risk [F5].
- Analyse lighting and image-quality effects.
- Note demographic limitations of FER-2013 honestly [F6].

## Before running

Read `../../../docs/system/MOOD_STATE_SPEC.md` — it defines the label space this model targets.

> **Note:** This notebook exists because benchmark accuracy does not transfer to phone use. Report limitations rather than concealing them.

## Status

Not yet run.


In [18]:
"""Run metadata — every training/evaluation notebook records its own run.

This is the project's experiment tracking (Gate 1A resolution): no separate
infrastructure, the notebook is responsible. Satisfies NFR-12.
"""
import json, os, time, random

RUN = {
    "run_id":          time.strftime("run_%Y%m%d_%H%M%S"),
    "timestamp":       time.strftime("%Y-%m-%d %H:%M:%S"),
    "notebook":        "06_error_analysis_gradcam",
    "dataset_version": None,   # set once data/processed/ is written
    "model_version":   None,   # set when a model is saved
    "hyperparameters": {},    # lr, batch_size, epochs, augmentation...
    "random_seed":     42,
    "metrics":         {},    # accuracy, macro_f1, per_class...
    "notes":           "",
}

random.seed(RUN["random_seed"])

def save_run(run=RUN, outdir="../outputs"):
    """Write the run record. Call at the END of the notebook."""
    os.makedirs(outdir, exist_ok=True)
    path = os.path.join(outdir, run["run_id"] + ".json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(run, f, indent=2)
    print("saved:", path)
    return path

RUN["run_id"]


'run_20260828_125554'

---

## Work starts here

## 0. Environment setup — repo root, DATA_ROOT (WSL2), artifact dirs

Same portability logic as notebooks 03/04/05: `find_repo_root()` never depends on CWD, and
`DATA_ROOT` prefers the WSL-native copy (`~/fer/data/raw`) over the `/mnt/c` repo copy,
warning loudly if it has to fall back. This notebook redefines the `RUN` dict from the
stub cell above with the fuller schema used by notebooks 03/04/05, so this run's record
has a comparable shape.

In [19]:
import os
import sys
import time

# --- repo root: walk up until ml/fer/notebooks is found ----------------------
def find_repo_root(start=None, max_up=8):
    """Auto-detect the repository root; never depends on the CWD being the notebook dir."""
    cur = os.path.abspath(start or os.getcwd())
    tried = []
    for _ in range(max_up):
        tried.append(cur)
        if os.path.isdir(os.path.join(cur, "ml", "fer", "notebooks")):
            return cur
        parent = os.path.dirname(cur)
        if parent == cur:
            break
        cur = parent
    raise FileNotFoundError(
        "Could not locate the repository root (a directory containing ml/fer/notebooks). "
        f"Directories tried, walking up from the CWD: {tried}"
    )

REPO_ROOT = find_repo_root()
FER_ROOT = os.path.join(REPO_ROOT, "ml", "fer")
print("REPO_ROOT:", REPO_ROOT)
print("FER_ROOT: ", FER_ROOT)
print("CWD:      ", os.getcwd())
print("Platform: ", sys.platform, "|", os.uname().release if hasattr(os, "uname") else "n/a")


# --- DATA_ROOT: prefer the WSL-native copy, fall back to the repo copy --------
def find_data_root(search_root, max_depth=3):
    """Return the directory that directly contains both 'train' and 'test'."""
    if not os.path.isdir(search_root):
        raise FileNotFoundError(f"search root does not exist: {search_root}")
    queue = [(search_root, 0)]
    visited = []
    while queue:
        current, depth = queue.pop(0)
        try:
            entries = os.listdir(current)
        except OSError:
            continue
        visited.append(current)
        lower = {e.lower(): e for e in entries}
        if "train" in lower and "test" in lower:
            tr = os.path.join(current, lower["train"])
            te = os.path.join(current, lower["test"])
            if os.path.isdir(tr) and os.path.isdir(te):
                return current
        if depth < max_depth:
            for e in entries:
                sub = os.path.join(current, e)
                if os.path.isdir(sub):
                    queue.append((sub, depth + 1))
    raise FileNotFoundError(
        f"no directory containing both 'train' and 'test' within {max_depth} levels of "
        f"{search_root}. Examined: {visited}"
    )

DATA_ROOT_CANDIDATES = [
    ("wsl-native", os.path.expanduser("~/fer/data/raw")),
    ("repo-mnt-c", os.path.join(FER_ROOT, "data", "raw")),
]

DATA_ROOT = None
DATA_ROOT_SOURCE = None
print()
print("Resolving DATA_ROOT (preference order: WSL-native, then the repo copy on /mnt/c):")
for _label, _cand in DATA_ROOT_CANDIDATES:
    print(f"  [{_label}] {_cand} -> exists={os.path.isdir(_cand)}")
    if DATA_ROOT is None and os.path.isdir(_cand):
        try:
            DATA_ROOT = find_data_root(_cand)
            DATA_ROOT_SOURCE = _label
        except FileNotFoundError as _e:
            print(f"      rejected: {_e}")

if DATA_ROOT is None:
    raise FileNotFoundError(
        "Neither candidate data root is usable. Copy the FER-2013 raw tree into the WSL native "
        "filesystem first:  mkdir -p ~/fer/data && cp -r "
        f"{os.path.join(FER_ROOT, 'data', 'raw')} ~/fer/data/"
    )

print()
print("Resolved DATA_ROOT:", DATA_ROOT, f"(source: {DATA_ROOT_SOURCE})")

SLOW_MOUNT = DATA_ROOT.startswith("/mnt/")
if SLOW_MOUNT:
    print()
    print("!" * 78)
    print("!!  WARNING - reading images from the WINDOWS MOUNT (/mnt/c).                !!")
    print("!!                                                                          !!")
    print("!!  Measured on this machine (see notebook 04): 800 JPEGs took 17.0 s from   !!")
    print("!!  /mnt/c versus 2.7 s from the WSL-native filesystem - a 6.4x penalty per   !!")
    print("!!  file, paid through the 9p/drvfs translation layer.                       !!")
    print("!!                                                                          !!")
    print("!!  This notebook reads pixels for lighting stats + runs one VAL inference   !!")
    print("!!  pass, so the penalty is paid at most once per image - but it is still     !!")
    print("!!  avoidable. Fix it with:                                                  !!")
    print("!!      mkdir -p ~/fer/data && cp -r <repo>/ml/fer/data/raw ~/fer/data/      !!")
    print("!" * 78)
else:
    print("Reading images from the WSL-native filesystem - no /mnt/c I/O penalty.")

# --- artifact directories: straight into the repo, no zip/download step ------
OUT_DIR    = os.path.join(FER_ROOT, "outputs")
PLOT_DIR   = os.path.join(FER_ROOT, "plots")
NB06_PLOT_DIR = os.path.join(PLOT_DIR, "nb06")
MODELS_DIR = os.path.join(FER_ROOT, "models")
for _d in (OUT_DIR, PLOT_DIR, NB06_PLOT_DIR, MODELS_DIR):
    os.makedirs(_d, exist_ok=True)
print()
print("OUT_DIR:      ", OUT_DIR)
print("PLOT_DIR:     ", PLOT_DIR)
print("NB06_PLOT_DIR:", NB06_PLOT_DIR)
print("MODELS_DIR:   ", MODELS_DIR)

# --- byte-budget tracker (same discipline as notebooks 02/03/04/05) ----------
_WRITTEN_FILES = []

def track_write(path, bucket="artifact"):
    """Record a file this notebook wrote, for the end-of-notebook byte report."""
    size = os.path.getsize(path)
    _WRITTEN_FILES.append((path, size))
    return size

def bucket_bytes(files):
    return int(sum(sz for _, sz in files))


REPO_ROOT: /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168
FER_ROOT:  /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer
CWD:       /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/notebooks
Platform:  linux | 6.6.87.2-microsoft-standard-WSL2

Resolving DATA_ROOT (preference order: WSL-native, then the repo copy on /mnt/c):
  [wsl-native] /home/yasinduslpredetor/fer/data/raw -> exists=True
  [repo-mnt-c] /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/data/raw -> exists=True

Resolved DATA_ROOT: /home/yasinduslpredetor/fer/data/raw (source: wsl-native)
Reading images from the WSL-native filesystem - no /mnt/c I/O penalty.

OUT_DIR:       /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/outputs
PLOT_DIR:      /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/plots
NB06_PLOT_DIR: /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/plots/nb06
MODELS_DIR:    /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer

In [20]:
import tensorflow as tf

print("TensorFlow:", tf.__version__, "| Keras:", tf.keras.__version__)

GPUS = tf.config.list_physical_devices("GPU")
print("tf.config.list_physical_devices('GPU'):", GPUS)

# This notebook runs ONE inference pass over VALIDATION (for Grad-CAM source
# predictions) - not a training loop - so a GPU is a speed convenience, not a
# hard requirement.
if GPUS:
    for _gpu in GPUS:
        try:
            tf.config.experimental.set_memory_growth(_gpu, True)
            print(f"  set_memory_growth(True) on {_gpu.name}")
        except RuntimeError as _e:
            print(f"  could not set memory growth on {_gpu.name}: {_e}")
    GPU_DEVICE_NAME = GPUS[0].name
    try:
        _details = tf.config.experimental.get_device_details(GPUS[0])
        GPU_DEVICE_DESCRIPTION = _details.get("device_name", "unknown")
    except Exception:
        GPU_DEVICE_DESCRIPTION = "unknown"
    print("GPU device name:       ", GPU_DEVICE_NAME)
    print("GPU device description:", GPU_DEVICE_DESCRIPTION)
    ENVIRONMENT = "wsl2-gpu"
else:
    print("No GPU visible - running on CPU. Tolerable: one pass over VALIDATION plus")
    print("Grad-CAM on a bounded, seeded sample of images per class.")
    GPU_DEVICE_NAME = None
    GPU_DEVICE_DESCRIPTION = None
    ENVIRONMENT = "wsl2-cpu"

print()
print(f"ENVIRONMENT = {ENVIRONMENT!r}")


TensorFlow: 2.21.0 | Keras: 3.15.1
tf.config.list_physical_devices('GPU'): [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
  set_memory_growth(True) on /physical_device:GPU:0
GPU device name:        /physical_device:GPU:0
GPU device description: NVIDIA GeForce RTX 3050 6GB Laptop GPU

ENVIRONMENT = 'wsl2-gpu'


In [21]:
"""Run metadata (redefinition) - the stub cell above is the project's fixed template;
this cell replaces its RUN/save_run with the fuller schema used by notebooks 03/04/05, so
this run's record has a comparable shape (package_versions, dataset_version as a content
hash, model_version, etc.).
"""
import json, os, time, random, sys, hashlib, importlib.metadata
import numpy as np

def _pkg_version(name, module=None, alt_dist_names=None):
    """Best-effort package version lookup; never raises."""
    for dist_name in [name] + list(alt_dist_names or []):
        try:
            return importlib.metadata.version(dist_name)
        except Exception:
            continue
    try:
        mod = module or __import__(name)
        return getattr(mod, "__version__", "unknown")
    except Exception:
        return "unknown"

PACKAGE_VERSIONS = {
    "python":       sys.version.split()[0],
    "numpy":        _pkg_version("numpy"),
    "pandas":       _pkg_version("pandas"),
    "Pillow":       _pkg_version("Pillow", module=__import__("PIL")),
    "matplotlib":   _pkg_version("matplotlib"),
    "seaborn":      _pkg_version("seaborn"),
    "scikit-learn": _pkg_version("scikit-learn", module=__import__("sklearn")),
    "scipy":        _pkg_version("scipy"),
    "tensorflow":   _pkg_version(
        "tensorflow", module=tf,
        alt_dist_names=["tensorflow-cpu", "tensorflow-gpu", "tensorflow-intel"],
    ),
    "keras":        _pkg_version("keras", module=tf.keras),
}
print("Package versions:")
for _k, _v in PACKAGE_VERSIONS.items():
    print(f"  {_k:<13} {_v}")

# --- dataset_version: manifest filename + SHA-256 computed at runtime --------
MANIFEST_PATH = os.path.join(OUT_DIR, "splits_cleaned.csv")
if not os.path.isfile(MANIFEST_PATH):
    raise FileNotFoundError(
        f"Manifest not found: {MANIFEST_PATH}. Run notebook 02 first (it writes splits_cleaned.csv)."
    )

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

MANIFEST_SHA256 = sha256_file(MANIFEST_PATH)
DATASET_VERSION = f"{os.path.basename(MANIFEST_PATH)}@sha256:{MANIFEST_SHA256}"
print()
print("Manifest:        ", MANIFEST_PATH)
print("Manifest SHA-256:", MANIFEST_SHA256)

RUN = {
    "run_id":           time.strftime("run_%Y%m%d_%H%M%S"),
    "timestamp":        time.strftime("%Y-%m-%d %H:%M:%S"),
    "notebook":         "06_error_analysis_gradcam",
    "dataset":          "FER-2013",
    "dataset_path":     DATA_ROOT,
    "dataset_root_source": DATA_ROOT_SOURCE,
    "dataset_version":  DATASET_VERSION,
    "model_version":    None,      # set below once the fine-tuned model is loaded
    "environment":      ENVIRONMENT,
    "gpu_device":       GPU_DEVICE_NAME,
    "gpu_description":  GPU_DEVICE_DESCRIPTION,
    "package_versions": PACKAGE_VERSIONS,
    "hyperparameters":  {},        # populated at the end from the actual analysis config
    "random_seed":      42,
    "metrics":          {},        # populated at the end from measured values only
    "notes":            "",
}

random.seed(RUN["random_seed"])
np.random.seed(RUN["random_seed"])
tf.keras.utils.set_random_seed(RUN["random_seed"])
RNG = np.random.default_rng(RUN["random_seed"])

print()
print(json.dumps({k: v for k, v in RUN.items() if k not in ("package_versions",)}, indent=2))

def save_run(run=RUN, outdir=None):
    """Write the run record. Safe to call repeatedly; the last call wins."""
    if outdir is None:
        outdir = OUT_DIR
    os.makedirs(outdir, exist_ok=True)
    path = os.path.join(outdir, run["run_id"] + ".json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(run, f, indent=2)
    print("saved:", path)
    return path

RUN["run_id"]


Package versions:
  python        3.11.15
  numpy         2.4.6
  pandas        3.0.5
  Pillow        12.3.0
  matplotlib    3.11.1
  seaborn       0.13.2
  scikit-learn  1.9.0
  scipy         1.17.1
  tensorflow    2.21.0
  keras         3.15.1

Manifest:         /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/outputs/splits_cleaned.csv
Manifest SHA-256: a75d460f35d14c14cba6830923e5e5483f6d5611e779a9252d0b57fd49325496

{
  "run_id": "run_20260828_125554",
  "timestamp": "2026-08-28 12:55:54",
  "notebook": "06_error_analysis_gradcam",
  "dataset": "FER-2013",
  "dataset_path": "/home/yasinduslpredetor/fer/data/raw",
  "dataset_root_source": "wsl-native",
  "dataset_version": "splits_cleaned.csv@sha256:a75d460f35d14c14cba6830923e5e5483f6d5611e779a9252d0b57fd49325496",
  "model_version": null,
  "environment": "wsl2-gpu",
  "gpu_device": "/physical_device:GPU:0",
  "gpu_description": "NVIDIA GeForce RTX 3050 6GB Laptop GPU",
  "hyperparameters": {},
  "random_seed": 42,
  "

'run_20260828_125554'

## 1. Load TEST predictions (fine-tuned model) — error characterisation

Reads `nb05_test_probabilities_finetuned.csv` only — **no re-inference on TEST anywhere
in this notebook**. Recreates `is_correct`, computes per-true-class error rate, and a
long-form confusion table (true_class, predicted_class, count, rate) sorted by count
descending among off-diagonal cells.

In [22]:
import pandas as pd

PROBS_PATH = os.path.join(OUT_DIR, "nb05_test_probabilities_finetuned.csv")
if not os.path.isfile(PROBS_PATH):
    raise FileNotFoundError(
        f"{PROBS_PATH} not found. Run notebook 05 first - it writes the fine-tuned model's "
        "unthresholded TEST softmax predictions, which this notebook reads (no re-inference "
        "on TEST is permitted here)."
    )

test_pred_df = pd.read_csv(PROBS_PATH)
print(f"Loaded {len(test_pred_df)} TEST predictions from {PROBS_PATH}")
print("Columns:", list(test_pred_df.columns))

assert (test_pred_df["model"] == "finetuned").all(), "expected only the finetuned model's rows"
assert test_pred_df["basename"].str.startswith("PrivateTest_").all(), \
    "non-TEST (non-PrivateTest_) rows leaked into the probabilities file"

# --- label space: SORTED class list, identical rule to notebooks 03/04/05 ----
CLASS_NAMES = sorted(test_pred_df["true_class"].unique().tolist())
N_CLASSES = len(CLASS_NAMES)
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
_EXPECTED_CLASS_ORDER = ["angry", "disgust", "fear", "happy", "neutral", "sad", "surprise"]
assert CLASS_NAMES == _EXPECTED_CLASS_ORDER, (
    f"class order does not match the expected alphabetical order used elsewhere in the "
    f"project: {CLASS_NAMES} != {_EXPECTED_CLASS_ORDER}"
)
print()
print("Class -> integer label mapping (sorted class list):")
for c in CLASS_NAMES:
    print(f"  {CLASS_TO_IDX[c]} -> {c}")

test_pred_df["is_correct"] = test_pred_df["true_label"] == test_pred_df["predicted_label"]
_overall_acc = float(test_pred_df["is_correct"].mean())
print()
print(f"Overall TEST accuracy (recomputed from CSV): {_overall_acc:.4f}")

# --- error rate per true class ------------------------------------------------
error_by_class = (
    test_pred_df.groupby("true_class")["is_correct"]
    .agg(n="count", accuracy="mean")
    .reindex(CLASS_NAMES)
)
error_by_class["error_rate"] = 1.0 - error_by_class["accuracy"]
print()
print("Error rate per true class:")
print(error_by_class.to_string(formatters={"accuracy": "{:.4f}".format,
                                            "error_rate": "{:.4f}".format}))

# --- long-form confusion table -------------------------------------------------
confusion_long = (
    test_pred_df.groupby(["true_class", "predicted_class"])
    .size()
    .reset_index(name="count")
)
_n_by_true = test_pred_df.groupby("true_class").size().rename("n_true")
confusion_long = confusion_long.merge(_n_by_true, left_on="true_class", right_index=True)
confusion_long["rate"] = confusion_long["count"] / confusion_long["n_true"]
confusion_long = confusion_long.drop(columns="n_true")

off_diag = confusion_long[confusion_long["true_class"] != confusion_long["predicted_class"]]
off_diag_sorted = off_diag.sort_values("count", ascending=False).reset_index(drop=True)

print()
print("Top 10 confusion pairs (off-diagonal, by count):")
print(off_diag_sorted.head(10).to_string(index=False, formatters={"rate": "{:.4f}".format}))


Loaded 3589 TEST predictions from /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/outputs/nb05_test_probabilities_finetuned.csv
Columns: ['basename', 'true_label', 'true_class', 'predicted_label', 'predicted_class', 'prob_angry', 'prob_disgust', 'prob_fear', 'prob_happy', 'prob_neutral', 'prob_sad', 'prob_surprise', 'model']

Class -> integer label mapping (sorted class list):
  0 -> angry
  1 -> disgust
  2 -> fear
  3 -> happy
  4 -> neutral
  5 -> sad
  6 -> surprise

Overall TEST accuracy (recomputed from CSV): 0.6289

Error rate per true class:
              n accuracy error_rate
true_class                         
angry       491   0.4969     0.5031
disgust      55   0.4909     0.5091
fear        528   0.4773     0.5227
happy       879   0.8783     0.1217
neutral     626   0.6262     0.3738
sad         594   0.4697     0.5303
surprise    416   0.6995     0.3005

Top 10 confusion pairs (off-diagonal, by count):
true_class predicted_class  count   rate
       sad      

## 2. Join to `image_flags.csv` on basename — BlazeFace confidence stratification

`image_flags.csv` carries `file_path` (Windows-backslash relative paths); the predictions
CSV carries `basename`. Both are reduced to a common basename and joined. The join is
asserted exact and complete — every TEST prediction row must match exactly one
`image_flags.csv` row, or this raises.

BlazeFace top-confidence is treated as a face-detection-quality proxy (see notebook 02 /
`docs/ml/02_data_preparation_findings.md`). Confidence is binned via `pd.qcut` (deciles,
falling back to fewer bins with `duplicates='drop'` if ties make deciles degenerate), and
accuracy / macro-F1 are computed per bin. A Spearman correlation between BlazeFace
confidence and per-sample correctness is used as the trend statistic — chosen because it
is monotonic-only and does not assume a linear relationship.

In [23]:
FLAGS_PATH = os.path.join(OUT_DIR, "image_flags.csv")
if not os.path.isfile(FLAGS_PATH):
    raise FileNotFoundError(f"image_flags.csv not found: {FLAGS_PATH}. Run notebook 02 first.")

flags_df = pd.read_csv(FLAGS_PATH)
flags_test = flags_df[flags_df["split_group"] == "TEST"].copy()
flags_test["basename"] = flags_test["file_path"].astype(str).str.replace("\\", "/", regex=False).apply(os.path.basename)

print(f"image_flags.csv TEST rows: {len(flags_test)}")
print(f"TEST predictions rows:     {len(test_pred_df)}")

_dupe_basenames = flags_test["basename"][flags_test["basename"].duplicated()]
if len(_dupe_basenames) > 0:
    raise AssertionError(
        f"image_flags.csv TEST basenames are not unique - {len(_dupe_basenames)} duplicates, "
        f"e.g. {_dupe_basenames.iloc[0]!r}. Cannot join exactly."
    )

merged = test_pred_df.merge(
    flags_test, on="basename", how="left", validate="one_to_one",
    suffixes=("", "_flags"), indicator="_join_status",
)

# Join completeness must be tested on the JOIN ITSELF, not on the nullability of a joined
# column. blazeface_top_confidence is legitimately NaN for images BlazeFace did not detect
# (notebook 02 measured 20 non-detections dataset-wide), so counting NaNs there conflates
# "row failed to join" with "value is genuinely missing".
_n_unmatched = int((merged["_join_status"] != "both").sum())
if _n_unmatched > 0 or len(merged) != len(test_pred_df):
    raise AssertionError(
        f"basename join between nb05_test_probabilities_finetuned.csv and image_flags.csv is "
        f"NOT exact/complete: {_n_unmatched} prediction rows failed to match a TEST row in "
        f"image_flags.csv (out of {len(test_pred_df)}). STOP - fix the join before trusting "
        "any stratified result below."
    )
merged.drop(columns=["_join_status"], inplace=True)
print(f"[PASS] join is exact and complete: {len(merged)} rows, "
      f"{len(merged) - _n_unmatched} matched, {_n_unmatched} unmatched.")

# --- BlazeFace non-detections: missing DATA, reported rather than silently dropped ------
_n_no_detection = int(merged["blazeface_top_confidence"].isna().sum())
print(f"TEST images with NO BlazeFace detection (confidence is NaN): {_n_no_detection}")
if _n_no_detection > 0:
    _nd = merged[merged["blazeface_top_confidence"].isna()]
    _nd_correct = int((_nd["true_class"] == _nd["predicted_class"]).sum())
    print(f"  of those, correctly classified by the FER model: {_nd_correct}/{_n_no_detection}")
    print("  (pd.qcut drops NaN, so these are excluded from the confidence deciles below;")
    print("   they are reported here explicitly as their own 'no-detection' stratum.)")
    for _b, _tc, _pc in _nd[["basename", "true_class", "predicted_class"]].head(10).itertuples(index=False):
        print(f"    {_b}  true={_tc}  pred={_pc}")

# --- decile binning of BlazeFace top confidence, degenerate-safe -------------
N_BINS_REQUESTED = 10
_conf = merged["blazeface_top_confidence"].astype(float)
_bf_bins, _bf_bin_edges = pd.qcut(_conf, q=N_BINS_REQUESTED, duplicates="drop", retbins=True)
merged["blazeface_conf_bin"] = _bf_bins
N_BF_BINS_ACTUAL = merged["blazeface_conf_bin"].nunique()
print()
print(f"Requested {N_BINS_REQUESTED} BlazeFace-confidence bins; actual distinct bins after "
      f"duplicates='drop': {N_BF_BINS_ACTUAL}")

from sklearn.metrics import f1_score, accuracy_score

def stratified_metrics(df, bin_col, order=None):
    rows = []
    groups = df.groupby(bin_col, observed=True)
    for bin_label, g in groups:
        _acc = float(accuracy_score(g["true_label"], g["predicted_label"]))
        _f1 = float(f1_score(g["true_label"], g["predicted_label"], average="macro", labels=list(range(N_CLASSES))))
        rows.append({"bin": str(bin_label), "n": int(len(g)), "accuracy": _acc, "macro_f1": _f1})
    out = pd.DataFrame(rows)
    if order == "as_seen":
        out["_ord"] = range(len(out))
    return out

bf_strat = stratified_metrics(merged, "blazeface_conf_bin")
# order bins by their interval's left edge (qcut bin labels are ordered categoricals)
_bin_order = list(merged["blazeface_conf_bin"].cat.categories.astype(str))
bf_strat["_rank"] = bf_strat["bin"].apply(lambda b: _bin_order.index(b))
bf_strat = bf_strat.sort_values("_rank").reset_index(drop=True)
bf_strat["bin_rank"] = range(len(bf_strat))

print()
print("Accuracy / macro-F1 by BlazeFace-confidence bin (low -> high confidence):")
print(bf_strat[["bin", "n", "accuracy", "macro_f1"]].to_string(
    index=False, formatters={"accuracy": "{:.4f}".format, "macro_f1": "{:.4f}".format}))

from scipy.stats import spearmanr

_spearman_conf_correct, _spearman_conf_correct_p = spearmanr(
    merged["blazeface_top_confidence"].astype(float), merged["is_correct"].astype(int)
)
print()
print(f"Spearman correlation (BlazeFace top confidence vs. per-sample correctness): "
      f"rho={_spearman_conf_correct:.4f}, p={_spearman_conf_correct_p:.4g}")
if _spearman_conf_correct > 0:
    print("Positive rho: correctness tends to rise as BlazeFace detection confidence rises "
          "(accuracy falls as confidence falls).")
else:
    print("Non-positive rho: no evidence that accuracy falls as BlazeFace confidence falls.")


image_flags.csv TEST rows: 3589
TEST predictions rows:     3589
[PASS] join is exact and complete: 3589 rows, 3589 matched, 0 unmatched.
TEST images with NO BlazeFace detection (confidence is NaN): 1
  of those, correctly classified by the FER model: 0/1
  (pd.qcut drops NaN, so these are excluded from the confidence deciles below;
   they are reported here explicitly as their own 'no-detection' stratum.)
    PrivateTest_57076415.jpg  true=neutral  pred=fear

Requested 10 BlazeFace-confidence bins; actual distinct bins after duplicates='drop': 10

Accuracy / macro-F1 by BlazeFace-confidence bin (low -> high confidence):
           bin   n accuracy macro_f1
(0.116, 0.543] 359   0.6128   0.6200
(0.543, 0.609] 359   0.5571   0.5702
 (0.609, 0.66] 359   0.6017   0.5691
   (0.66, 0.7] 358   0.6201   0.5721
  (0.7, 0.739] 359   0.6156   0.5681
 (0.739, 0.77] 359   0.6546   0.5942
 (0.77, 0.801] 358   0.6564   0.5800
(0.801, 0.832] 359   0.6657   0.6030
(0.832, 0.867] 359   0.6490   0.5584
(0

In [24]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax1 = plt.subplots(figsize=(9, 5))
ax1.plot(bf_strat["bin_rank"], bf_strat["accuracy"], marker="o", color="#4C72B0", label="accuracy")
ax1.plot(bf_strat["bin_rank"], bf_strat["macro_f1"], marker="s", color="#DD8452", label="macro-F1")
ax1.set_xlabel("BlazeFace-confidence bin (0 = lowest confidence decile, "
               f"{N_BF_BINS_ACTUAL - 1} = highest)")
ax1.set_ylabel("score")
ax1.set_ylim(0, 1)
ax1.set_title("TEST accuracy / macro-F1 by BlazeFace detection-confidence bin\n"
              f"(fine-tuned model; Spearman rho(conf, correct)={_spearman_conf_correct:.3f})")
ax1.legend()
ax1.grid(alpha=0.3)
fig.tight_layout()
_p = os.path.join(NB06_PLOT_DIR, "accuracy_by_blazeface_confidence.png")
fig.savefig(_p, dpi=150)
plt.close(fig)
track_write(_p)
print("saved:", _p)


saved: /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/plots/nb06/accuracy_by_blazeface_confidence.png


## 3. Lighting — per-image pixel intensity (TEST, pixel-only, no model inference)

Per-image mean and std pixel intensity is a descriptive statistic computed directly from
the raw grayscale pixels on disk — not a model inference — so it is permitted on TEST.
Images are read from `DATA_ROOT` (WSL-native, `/mnt/c` fallback) exactly like notebook 05's
pipeline, but decoded with PIL directly (no model preprocessing needed for a pixel
statistic). Intensity is binned into deciles and accuracy / macro-F1 computed per bin.

In [25]:
from PIL import Image

def rebuild_test_path(basename):
    """TEST images all live under DATA_ROOT/test/<class>/<basename>."""
    return None  # placeholder replaced below once we know the class per basename

# basename -> class, from image_flags.csv (already joined for TEST rows above)
_basename_to_class = dict(zip(merged["basename"], merged["class"]))

def abs_test_path(basename):
    cls = _basename_to_class[basename]
    return os.path.join(DATA_ROOT, "test", str(cls), basename)

merged["abs_path"] = merged["basename"].apply(abs_test_path)

_missing = [p for p in merged["abs_path"] if not os.path.isfile(p)]
if _missing:
    raise FileNotFoundError(
        f"{len(_missing)} rebuilt TEST paths are missing from disk. First: {_missing[0]}"
    )
print(f"[PASS] all {len(merged)} rebuilt TEST paths exist on disk.")

print("Reading raw grayscale pixel statistics for all TEST images (pixel-level only, "
      "no model inference) ...")
_t0 = time.time()
_means, _stds = [], []
for _p in merged["abs_path"]:
    with Image.open(_p) as _img:
        _arr = np.asarray(_img.convert("L"), dtype=np.float32)
    _means.append(float(_arr.mean()))
    _stds.append(float(_arr.std()))
merged["pixel_mean_intensity"] = _means
merged["pixel_std_intensity"] = _stds
print(f"  done in {time.time() - _t0:.1f}s over {len(merged)} images")

N_INTENSITY_BINS_REQUESTED = 10
_int_bins, _int_bin_edges = pd.qcut(
    merged["pixel_mean_intensity"], q=N_INTENSITY_BINS_REQUESTED, duplicates="drop", retbins=True
)
merged["intensity_bin"] = _int_bins
N_INTENSITY_BINS_ACTUAL = merged["intensity_bin"].nunique()
print(f"Requested {N_INTENSITY_BINS_REQUESTED} intensity bins; actual distinct bins: "
      f"{N_INTENSITY_BINS_ACTUAL}")

intensity_strat = stratified_metrics(merged, "intensity_bin")
_int_bin_order = list(merged["intensity_bin"].cat.categories.astype(str))
intensity_strat["_rank"] = intensity_strat["bin"].apply(lambda b: _int_bin_order.index(b))
intensity_strat = intensity_strat.sort_values("_rank").reset_index(drop=True)
intensity_strat["bin_rank"] = range(len(intensity_strat))

print()
print("Accuracy / macro-F1 by pixel-intensity bin (dark -> bright):")
print(intensity_strat[["bin", "n", "accuracy", "macro_f1"]].to_string(
    index=False, formatters={"accuracy": "{:.4f}".format, "macro_f1": "{:.4f}".format}))

_spearman_intensity_correct, _spearman_intensity_correct_p = spearmanr(
    merged["pixel_mean_intensity"], merged["is_correct"].astype(int)
)
print()
print(f"Spearman correlation (mean pixel intensity vs. per-sample correctness): "
      f"rho={_spearman_intensity_correct:.4f}, p={_spearman_intensity_correct_p:.4g}")


[PASS] all 3589 rebuilt TEST paths exist on disk.
Reading raw grayscale pixel statistics for all TEST images (pixel-level only, no model inference) ...
  done in 0.4s over 3589 images
Requested 10 intensity bins; actual distinct bins: 10

Accuracy / macro-F1 by pixel-intensity bin (dark -> bright):
               bin   n accuracy macro_f1
  (14.311, 85.332] 359   0.5989   0.4574
 (85.332, 100.537] 359   0.5933   0.5367
(100.537, 111.474] 359   0.5710   0.5444
(111.474, 120.688] 360   0.5889   0.5706
(120.688, 129.322] 358   0.6508   0.6158
 (129.322, 137.58] 358   0.6425   0.6544
 (137.58, 146.917] 359   0.6685   0.5865
(146.917, 158.148] 359   0.6100   0.5926
(158.148, 172.374] 359   0.6657   0.5418
(172.374, 244.806] 359   0.6992   0.6435

Spearman correlation (mean pixel intensity vs. per-sample correctness): rho=0.0681, p=4.455e-05


In [26]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(intensity_strat["bin_rank"], intensity_strat["accuracy"], marker="o", color="#4C72B0", label="accuracy")
ax.plot(intensity_strat["bin_rank"], intensity_strat["macro_f1"], marker="s", color="#DD8452", label="macro-F1")
ax.set_xlabel(f"pixel-intensity bin (0 = darkest decile, {N_INTENSITY_BINS_ACTUAL - 1} = brightest)")
ax.set_ylabel("score")
ax.set_ylim(0, 1)
ax.set_title("TEST accuracy / macro-F1 by mean pixel-intensity bin\n"
             f"(Spearman rho(intensity, correct)={_spearman_intensity_correct:.3f})")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
_p = os.path.join(NB06_PLOT_DIR, "accuracy_by_intensity_decile.png")
fig.savefig(_p, dpi=150)
plt.close(fig)
track_write(_p)
print("saved:", _p)


saved: /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/plots/nb06/accuracy_by_intensity_decile.png


## 4. Brightness confound — does predicted class track brightness beyond true class?

Method: compute the mean pixel intensity per **true** class (baseline expectation) and per
**predicted** class. For **misclassified** TEST samples only, compare
`|intensity - mean_intensity[predicted_class]|` against
`|intensity - mean_intensity[true_class]|`. If the predicted class's typical brightness is
the closer match more often than a random/null assumption would predict, that is evidence
the model leans on brightness rather than expression content for some errors.

**Limitation, stated plainly:** this is a correlational check, not a causal proof. The
per-class sample sizes among misclassifications are small, and brightness is itself
confounded with expression content in the source photography (e.g. certain expressions may
correlate with lighting conditions in how FER-2013 was collected) — so a "predicted-class
brightness is closer" result does not by itself prove the model is using brightness as a
shortcut, only that it is *consistent* with that hypothesis. If the result is strong, it is
reported as an escalation trigger, not softened.

In [27]:
mean_intensity_by_true_class = merged.groupby("true_class")["pixel_mean_intensity"].mean()
mean_intensity_by_pred_class = merged.groupby("predicted_class")["pixel_mean_intensity"].mean()

print("Mean pixel intensity by TRUE class:")
print(mean_intensity_by_true_class.reindex(CLASS_NAMES).to_string(float_format="{:.2f}".format))
print()
print("Mean pixel intensity by PREDICTED class:")
print(mean_intensity_by_pred_class.reindex(CLASS_NAMES).to_string(float_format="{:.2f}".format))

misclassified = merged[~merged["is_correct"]].copy()
print()
print(f"Misclassified TEST samples: {len(misclassified)} / {len(merged)}")

misclassified["dist_to_true_class_intensity"] = (
    misclassified["pixel_mean_intensity"] - misclassified["true_class"].map(mean_intensity_by_true_class)
).abs()
misclassified["dist_to_pred_class_intensity"] = (
    misclassified["pixel_mean_intensity"] - misclassified["predicted_class"].map(mean_intensity_by_pred_class)
).abs()
misclassified["closer_to_predicted"] = (
    misclassified["dist_to_pred_class_intensity"] < misclassified["dist_to_true_class_intensity"]
)

_frac_closer_to_predicted = float(misclassified["closer_to_predicted"].mean())
_null_expectation = 0.5  # under a null/random-confusion assumption, a misclassified sample's
                          # brightness is equally likely to be closer to either class's mean
print()
print(f"Fraction of misclassified samples whose brightness is closer to the PREDICTED class's "
      f"mean than to the TRUE class's mean: {_frac_closer_to_predicted:.4f}")
print(f"Null/random-confusion expectation: {_null_expectation:.4f}")
_confound_delta = _frac_closer_to_predicted - _null_expectation
print(f"Delta vs. null: {_confound_delta:+.4f}")

if _confound_delta > 0.05:
    print()
    print("[ESCALATION] Misclassified samples lean toward the predicted class's brightness "
          "profile more often than chance. This is CONSISTENT with the model exploiting "
          "brightness as a shortcut for some errors - it is not proof, but it should not be "
          "softened or omitted from reporting.")
elif _confound_delta < -0.05:
    print()
    print("Misclassified samples lean AWAY from the predicted class's brightness profile - "
          "no evidence of a brightness-shortcut confound from this test.")
else:
    print()
    print("Result is close to the null expectation - no strong evidence either way from this "
          "test alone.")


Mean pixel intensity by TRUE class:
true_class
angry      128.51
disgust    133.43
fear       134.81
happy      127.67
neutral    123.44
sad        120.04
surprise   146.28

Mean pixel intensity by PREDICTED class:
predicted_class
angry      128.47
disgust    134.70
fear       137.04
happy      127.42
neutral    124.31
sad        116.39
surprise   149.54

Misclassified TEST samples: 1332 / 3589

Fraction of misclassified samples whose brightness is closer to the PREDICTED class's mean than to the TRUE class's mean: 0.5150
Null/random-confusion expectation: 0.5000
Delta vs. null: +0.0150

Result is close to the null expectation - no strong evidence either way from this test alone.


## 5. Calibration — reliability diagram and Expected Calibration Error (ECE)

Predicted max-softmax confidence is binned into 10 equal-width bins `[0,1]`. Per bin:
empirical accuracy vs. mean confidence. ECE = `sum_bins (bin_count/N) * |bin_accuracy -
bin_mean_confidence|`. No threshold (`tau_face_min`) is selected here — only calibration is
reported.

In [28]:
_prob_cols = [f"prob_{c}" for c in CLASS_NAMES]
merged["max_confidence"] = merged[_prob_cols].max(axis=1)

N_CAL_BINS = 10
_cal_bin_edges = np.linspace(0.0, 1.0, N_CAL_BINS + 1)
_cal_bin_idx = np.clip(np.digitize(merged["max_confidence"], _cal_bin_edges[1:-1], right=True), 0, N_CAL_BINS - 1)
merged["cal_bin"] = _cal_bin_idx

cal_rows = []
N = len(merged)
ece = 0.0
for b in range(N_CAL_BINS):
    g = merged[merged["cal_bin"] == b]
    n_b = len(g)
    if n_b == 0:
        cal_rows.append({"bin": b, "mean_confidence": np.nan, "empirical_accuracy": np.nan, "n": 0})
        continue
    _mean_conf = float(g["max_confidence"].mean())
    _emp_acc = float(g["is_correct"].mean())
    cal_rows.append({"bin": b, "mean_confidence": _mean_conf, "empirical_accuracy": _emp_acc, "n": n_b})
    ece += (n_b / N) * abs(_emp_acc - _mean_conf)

calibration_df = pd.DataFrame(cal_rows)
ECE = float(ece)
print("Per-bin calibration:")
print(calibration_df.to_string(index=False, formatters={
    "mean_confidence": "{:.4f}".format, "empirical_accuracy": "{:.4f}".format}))
print()
print(f"Expected Calibration Error (ECE) = {ECE:.4f}")


Per-bin calibration:
 bin mean_confidence empirical_accuracy    n
   0             NaN                NaN    0
   1             NaN                NaN    0
   2             NaN                NaN    0
   3          0.3580             0.4286    7
   4          0.4617             0.2449   49
   5          0.5461             0.3281  128
   6          0.6513             0.3743  171
   7          0.7537             0.3620  163
   8          0.8509             0.4128  235
   9          0.9894             0.6982 2836

Expected Calibration Error (ECE) = 0.3006


In [29]:
fig, ax = plt.subplots(figsize=(6.5, 6.5))
_valid = calibration_df.dropna(subset=["mean_confidence", "empirical_accuracy"])
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="perfect calibration")
ax.plot(_valid["mean_confidence"], _valid["empirical_accuracy"], marker="o", color="#4C72B0",
        label="fine-tuned model (TEST)")
ax.set_xlabel("mean predicted confidence (bin)")
ax.set_ylabel("empirical accuracy (bin)")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title(f"Reliability diagram - fine-tuned model, TEST\nECE = {ECE:.4f}")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
_p = os.path.join(NB06_PLOT_DIR, "reliability_diagram.png")
fig.savefig(_p, dpi=150)
plt.close(fig)
track_write(_p)
print("saved:", _p)


saved: /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/plots/nb06/reliability_diagram.png


## 6. Export Section A artifacts to `../outputs/`

- `nb06_error_analysis.csv` — long-form confusion table (true_class, predicted_class,
  count, rate), all cells (diagonal + off-diagonal).
- `nb06_stratified_accuracy.csv` — both stratifications (BlazeFace confidence and pixel
  intensity), distinguished by a `stratifier` column.
- `nb06_calibration.csv` — per-bin calibration table plus an ECE summary row.

In [30]:
written_before_A_export = bucket_bytes(_WRITTEN_FILES)

# --- (a) nb06_error_analysis.csv ----------------------------------------------
p_err = os.path.join(OUT_DIR, "nb06_error_analysis.csv")
confusion_long.to_csv(p_err, index=False)
track_write(p_err)
print("saved:", p_err, f"({len(confusion_long)} rows)")

# --- (b) nb06_stratified_accuracy.csv -----------------------------------------
_bf_out = bf_strat[["bin", "n", "accuracy", "macro_f1"]].copy()
_bf_out.insert(0, "stratifier", "blazeface_top_confidence")
_int_out = intensity_strat[["bin", "n", "accuracy", "macro_f1"]].copy()
_int_out.insert(0, "stratifier", "pixel_mean_intensity")
stratified_accuracy_df = pd.concat([_bf_out, _int_out], ignore_index=True)
p_strat = os.path.join(OUT_DIR, "nb06_stratified_accuracy.csv")
stratified_accuracy_df.to_csv(p_strat, index=False)
track_write(p_strat)
print("saved:", p_strat, f"({len(stratified_accuracy_df)} rows)")

# --- (c) nb06_calibration.csv --------------------------------------------------
_cal_out = calibration_df.copy()
_cal_out["ece"] = None
_cal_out.loc[len(_cal_out)] = {"bin": "ECE_SUMMARY", "mean_confidence": None,
                                "empirical_accuracy": None, "n": int(N), "ece": ECE}
p_cal = os.path.join(OUT_DIR, "nb06_calibration.csv")
_cal_out.to_csv(p_cal, index=False)
track_write(p_cal)
print("saved:", p_cal, f"({len(_cal_out)} rows, incl. ECE summary row)")

written_after_A_export = bucket_bytes(_WRITTEN_FILES)
print()
print(f"Section A artifact bytes: {written_after_A_export - written_before_A_export}")


saved: /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/outputs/nb06_error_analysis.csv (48 rows)
saved: /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/outputs/nb06_stratified_accuracy.csv (20 rows)
saved: /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/outputs/nb06_calibration.csv (11 rows, incl. ECE summary row)

Section A artifact bytes: 3864


## 7. VALIDATION `tf.data` pipeline — identical preprocessing to nb03/04/05

This is the **only** place in this notebook that re-runs the model, and it runs on the
**VALIDATION** split (`split_group == "VAL"`), never TEST. The exact validation prefix is
verified at runtime rather than assumed.

In [31]:
manifest_full = pd.read_csv(MANIFEST_PATH)
print("split_group counts:")
print(manifest_full["split_group"].value_counts().to_string())

val_df = manifest_full[manifest_full["split_group"] == "VAL"].reset_index(drop=True).copy()

_val_prefixes = val_df["prefix"].unique().tolist()
print()
print(f"VAL split prefixes found: {_val_prefixes}")
if len(_val_prefixes) != 1:
    raise AssertionError(
        f"VAL split has more than one distinct prefix: {_val_prefixes} - expected exactly one."
    )
VAL_PREFIX = _val_prefixes[0]
print(f"VAL_PREFIX (verified at runtime) = {VAL_PREFIX!r}")

def rebuild_path(file_path, split_group, cls):
    split_dir = "train" if split_group == "TRAIN" else "test"
    basename = os.path.basename(str(file_path).replace("\\", "/"))
    return os.path.join(DATA_ROOT, split_dir, str(cls), basename)

val_df["basename"] = [os.path.basename(str(p).replace("\\", "/")) for p in val_df["file_path"]]
val_df["abs_path"] = [
    rebuild_path(p, g, c) for p, g, c in zip(val_df["file_path"], val_df["split_group"], val_df["class"])
]

_missing = [p for p in val_df["abs_path"] if not os.path.isfile(p)]
if _missing:
    raise FileNotFoundError(f"{len(_missing)} rebuilt VAL paths missing. First: {_missing[0]}")
print(f"[PASS] all {len(val_df)} rebuilt VAL paths exist on disk.")

val_class_names = sorted(val_df["class"].unique().tolist())
assert val_class_names == CLASS_NAMES, (
    f"VAL class set {val_class_names} does not match TEST-derived CLASS_NAMES {CLASS_NAMES}"
)

val_paths = val_df["abs_path"].tolist()
val_basenames = val_df["basename"].tolist()
y_val = np.array([CLASS_TO_IDX[c] for c in val_df["class"]], dtype=np.int64)
print(f"val_paths: {len(val_paths)}  y_val: {y_val.shape}")

def assert_val_only(*path_lists, label=""):
    bad = []
    for plist in path_lists:
        for p in plist:
            b = os.path.basename(str(p).replace("\\", "/"))
            if not b.startswith(VAL_PREFIX):
                bad.append(p)
    if bad:
        raise AssertionError(
            f"NON-VAL FILE LEAKED{' in ' + label if label else ''}: {len(bad)} files not "
            f"prefixed {VAL_PREFIX!r}. First: {bad[0]}"
        )
    return True

assert_val_only(val_paths, label="section 7 path arrays")
print(f"[PASS] every path in val_paths starts with {VAL_PREFIX!r}.")


split_group counts:
split_group
TRAIN    26901
TEST      3589
VAL       3589

VAL split prefixes found: ['PublicTest_']
VAL_PREFIX (verified at runtime) = 'PublicTest_'
[PASS] all 3589 rebuilt VAL paths exist on disk.
val_paths: 3589  y_val: (3589,)
[PASS] every path in val_paths starts with 'PublicTest_'.


In [32]:
AUTOTUNE = tf.data.AUTOTUNE
NATIVE_SIZE = 48          # FER-2013 native resolution; NOT a model input size
INPUT_SIZE  = 96          # model input, inherited from notebook 03's decision
EVAL_BATCH_SIZE = 64      # matches notebooks 04/05

def _decode_and_preprocess(path):
    raw = tf.io.read_file(path)
    img = tf.io.decode_jpeg(raw, channels=1, dct_method="INTEGER_ACCURATE")
    # dct_method pinned: bit-exact with PIL (see notebook 04).
    img = tf.image.resize(img, (NATIVE_SIZE, NATIVE_SIZE), method="bilinear")
    img = tf.cast(tf.round(img), tf.uint8)                              # (48, 48, 1) uint8
    x = tf.cast(img, tf.float32)
    x = tf.image.grayscale_to_rgb(x)                                    # (48, 48, 3)
    x = tf.image.resize(x, (INPUT_SIZE, INPUT_SIZE), method="bilinear")  # (96, 96, 3)
    x = tf.keras.applications.mobilenet_v2.preprocess_input(x)          # -> [-1, 1]
    return x

def make_eval_dataset(paths, labels, batch_size=EVAL_BATCH_SIZE):
    ds = tf.data.Dataset.from_tensor_slices(
        (tf.constant(paths, dtype=tf.string), tf.constant(labels, dtype=tf.int64))
    )
    ds = ds.map(lambda p, y: (_decode_and_preprocess(p), y), num_parallel_calls=AUTOTUNE, deterministic=True)
    ds = ds.batch(batch_size).prefetch(AUTOTUNE)
    return ds

VAL_DS = make_eval_dataset(val_paths, y_val)

_xb, _yb = next(iter(VAL_DS))
print("VAL batch: x", _xb.shape, _xb.dtype,
      f"range [{float(tf.reduce_min(_xb)):.3f}, {float(tf.reduce_max(_xb)):.3f}]", " y", _yb.shape)
assert tuple(_xb.shape[1:]) == (INPUT_SIZE, INPUT_SIZE, 3), "unexpected VAL batch shape"
del _xb, _yb

_val_labels_from_ds = np.concatenate([yb.numpy() for _, yb in VAL_DS], axis=0)
assert np.array_equal(_val_labels_from_ds, y_val), "VAL dataset re-orders rows"
del _val_labels_from_ds
print(f"[PASS] VAL_DS ready: {len(val_paths)} images, batch_size={EVAL_BATCH_SIZE}.")


VAL batch: x (64, 96, 96, 3) <dtype: 'float32'> range [-1.000, 1.000]  y (64,)
[PASS] VAL_DS ready: 3589 images, batch_size=64.


## 8. Load the fine-tuned model and run inference ONCE over VALIDATION

This is the only inference pass in the notebook. Running it on VALIDATION (never TEST) is
permitted and necessary to identify which validation samples are correctly/incorrectly
classified for Grad-CAM sampling.

In [33]:
MODEL_PATH = os.path.join(MODELS_DIR, "fer_mobilenetv2_finetuned_96.keras")
if not os.path.isfile(MODEL_PATH):
    raise FileNotFoundError(f"model file not found: {MODEL_PATH}. Run notebook 04 first.")

print(f"Loading fine-tuned model: {MODEL_PATH}")
_t0 = time.time()
MODEL = tf.keras.models.load_model(MODEL_PATH)
print(f"  loaded in {time.time() - _t0:.1f}s")
RUN["model_version"] = os.path.basename(MODEL_PATH)

assert_val_only(val_paths, label="section 8 inference input")
_t0 = time.time()
VAL_PROBS = MODEL.predict(VAL_DS, verbose=0).astype(np.float32)
print(f"VAL inference done in {time.time() - _t0:.1f}s over {len(val_paths)} images.")
assert VAL_PROBS.shape == (len(val_paths), N_CLASSES), f"unexpected VAL_PROBS shape {VAL_PROBS.shape}"
assert np.allclose(VAL_PROBS.sum(axis=1), 1.0, atol=1e-3), "VAL softmax rows do not sum to 1"
VAL_PREDS = VAL_PROBS.argmax(axis=1)

val_result_df = pd.DataFrame({
    "basename": val_basenames,
    "abs_path": val_paths,
    "true_label": y_val,
    "true_class": [CLASS_NAMES[i] for i in y_val],
    "predicted_label": VAL_PREDS,
    "predicted_class": [CLASS_NAMES[i] for i in VAL_PREDS],
})
val_result_df["is_correct"] = val_result_df["true_label"] == val_result_df["predicted_label"]
_val_acc = float(val_result_df["is_correct"].mean())
print(f"VALIDATION accuracy (this run, informational only): {_val_acc:.4f}")


Loading fine-tuned model: /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/models/fer_mobilenetv2_finetuned_96.keras
  loaded in 1.5s
VAL inference done in 5.2s over 3589 images.
VALIDATION accuracy (this run, informational only): 0.6319


## 9. Grad-CAM implementation — last convolutional layer of the MobileNetV2 backbone

The last conv layer is located **at runtime** by walking `model.layers` (and, since
MobileNetV2 is loaded as a nested sub-model/functional graph, its inner layers too),
looking for the last layer whose output has rank 4 (`(batch, H, W, C)`). A commented
fallback name (`"Conv_1"`, MobileNetV2's usual last conv) is left only as documentation,
never used as the primary lookup.

In [34]:
def _rank4_output_shape(layer):
    """Return the layer's output shape if it is rank-4, else None.

    Keras 3 REMOVED `Layer.output_shape`; the shape now comes from the output tensor
    (`layer.output.shape`). The previous implementation read `layer.output_shape` inside a
    bare `except Exception: continue`, which silently swallowed the AttributeError for
    EVERY layer and produced an empty candidate list. The `.output_shape` fallback below is
    retained only for older Keras.
    """
    for _getter in (lambda l: tuple(l.output.shape), lambda l: l.output_shape):
        try:
            shp = _getter(layer)
        except Exception:
            continue
        if isinstance(shp, tuple) and len(shp) == 4:
            return shp
    return None


def find_backbone_and_last_conv_layer(model):
    """Depth-first search for the last layer (anywhere in the nested model graph) whose
    output shape is rank-4 (a feature-map-producing conv/pooling-eligible layer).

    MobileNetV2 is very likely nested as a sub-model inside the Sequential/Functional
    top-level `model` (notebooks 03/04's transfer-learning pattern loads
    `tf.keras.applications.MobileNetV2` as one layer of the outer model), so we must recurse
    into any layer that itself exposes `.layers`.
    """
    candidates = []  # (owning_model, layer, depth)

    def _walk(m, depth=0):
        for layer in m.layers:
            sub_layers = getattr(layer, "layers", None)
            if sub_layers:
                _walk(layer, depth + 1)
            else:
                shape = _rank4_output_shape(layer)
                if shape is not None:
                    candidates.append((m, layer, depth))

    _walk(model)
    if not candidates:
        raise RuntimeError(
            "Could not find any layer with a rank-4 output anywhere in the model graph - "
            "Grad-CAM requires a convolutional feature map. Inspect model.summary() manually."
        )
    # last rank-4-output layer encountered in a depth-first walk is taken as "last conv
    # layer of the backbone" (fallback reference only, never used as primary lookup:
    # MobileNetV2's canonical last conv layer name is "Conv_1")
    owning_model, last_conv_layer, _depth = candidates[-1]
    return owning_model, last_conv_layer

GRADCAM_OWNING_MODEL, GRADCAM_LAST_CONV_LAYER = find_backbone_and_last_conv_layer(MODEL)
print(f"Located last conv/feature-map layer: name={GRADCAM_LAST_CONV_LAYER.name!r}, "
      f"type={type(GRADCAM_LAST_CONV_LAYER).__name__}, "
      f"output_shape={_rank4_output_shape(GRADCAM_LAST_CONV_LAYER)}, "
      f"owning sub-model={getattr(GRADCAM_OWNING_MODEL, 'name', 'top-level')!r}")


Located last conv/feature-map layer: name='out_relu', type=ReLU, output_shape=(None, 3, 3, 1280), owning sub-model='mobilenetv2_1.00_96'


In [35]:
def build_gradcam_model(full_model, owning_model, conv_layer):
    """A functional model mapping full_model's input to (conv_layer output, full_model output).

    Even when conv_layer lives inside a nested sub-model (e.g. MobileNetV2 loaded as one
    layer of the outer transfer-learning model), `conv_layer.output` is still a valid
    symbolic tensor within the OUTER graph, as long as the layer was called exactly once
    when the outer model was built. Keras's functional API resolves the full computation
    graph through nested calls, so a single `tf.keras.Model(inputs=full_model.inputs,
    outputs=[conv_layer.output, full_model.output])` works directly in both the flat and
    nested case - no separate two-stage GradientTape is needed (an earlier draft of this
    attempted that for the nested case, but it built `preds` from an independent forward
    pass through `full_model`, disconnected from `conv_out`, which cannot yield gradients
    with a plain `GradientTape` - `tape.watch` alone does not create a graph dependency).
    """
    if owning_model is not full_model:
        # NESTED case (this project's models: MobileNetV2 is one layer of the outer model).
        # conv_layer.output belongs to the SUB-model's graph, not the outer one. In Keras 3
        # a single functional model across that boundary raises
        #   ValueError: Output with path `0` is not connected to `inputs`
        # (verified against fer_mobilenetv2_finetuned_96.keras). Build the grad model from
        # the SUB-model and replay the remaining head layers inside the tape instead.
        sub = tf.keras.models.Model(
            owning_model.inputs, [conv_layer.output, owning_model.output]
        )
        _idx = full_model.layers.index(owning_model)
        head_layers = full_model.layers[_idx + 1:]
        return (sub, head_layers), "nested_two_stage"

    grad_model = tf.keras.models.Model(
        inputs=full_model.inputs, outputs=[conv_layer.output, full_model.output]
    )
    return (grad_model, None), "single_stage"

GRAD_MODEL, GRADCAM_MODE = build_gradcam_model(MODEL, GRADCAM_OWNING_MODEL, GRADCAM_LAST_CONV_LAYER)
print(f"Grad-CAM mode: {GRADCAM_MODE}")

def compute_gradcam(image_batch, class_index):
    """Return a (H, W) heatmap in [0, 1] for one image (image_batch shape (1, H, W, 3))."""
    _model_obj, _head_layers = GRAD_MODEL
    with tf.GradientTape() as tape:
        conv_out, _tail = _model_obj(image_batch, training=False)
        if _head_layers is not None:
            preds = _tail
            for _l in _head_layers:
                preds = _l(preds, training=False)
        else:
            preds = _tail
        loss = preds[:, class_index]
    grads = tape.gradient(loss, conv_out)

    if grads is None:
        raise RuntimeError(
            "Gradient of the class score w.r.t. the conv layer output is None - Grad-CAM "
            "could not connect gradients through the model graph. Inspect model.summary() "
            "/ the layer wiring, or confirm the located conv layer is actually upstream of "
            "the output."
        )

    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))          # (C,)
    conv_out_0 = conv_out[0]                                       # (H, W, C)
    heatmap = tf.reduce_sum(conv_out_0 * pooled_grads, axis=-1)    # (H, W)
    heatmap = tf.maximum(heatmap, 0.0)
    _max = tf.reduce_max(heatmap)
    if _max > 0:
        heatmap = heatmap / _max
    return heatmap.numpy()


Grad-CAM mode: nested_two_stage


## 10. Grad-CAM per class — averaged heatmaps (up to 100 correctly-classified VAL images/class)

For each of the 7 classes, up to 100 correctly-classified validation images are sampled
(seeded, `np.random.default_rng(42)`; fewer if not available — actual count is reported).
Grad-CAM heatmaps are computed per image, resized to 96x96, and averaged into one heatmap
per class. Only the averaged heatmap is plotted (no face image beneath it) — this keeps
the committable plot free of any identifiable individual.

In [36]:
N_GRADCAM_SAMPLES_PER_CLASS = 100

def load_and_preprocess_single(path):
    x = _decode_and_preprocess(tf.constant(path))
    return tf.expand_dims(x, axis=0)  # (1, H, W, 3)

def resize_heatmap(heatmap, size=INPUT_SIZE):
    h = tf.image.resize(heatmap[..., np.newaxis], (size, size), method="bilinear")
    return h.numpy()[..., 0]

per_class_avg_heatmap = {}
per_class_sample_count = {}

for _cls in CLASS_NAMES:
    _cls_idx = CLASS_TO_IDX[_cls]
    _correct_val = val_result_df[
        (val_result_df["true_class"] == _cls) & (val_result_df["is_correct"])
    ]
    _n_available = len(_correct_val)
    _n_take = min(N_GRADCAM_SAMPLES_PER_CLASS, _n_available)
    if _n_available == 0:
        print(f"  [{_cls}] no correctly-classified VAL samples available - skipping.")
        per_class_avg_heatmap[_cls] = None
        per_class_sample_count[_cls] = 0
        continue
    _idx = RNG.choice(_n_available, size=_n_take, replace=False)
    _sampled_paths = _correct_val["abs_path"].to_numpy()[_idx]

    _heatmaps = []
    for _p in _sampled_paths:
        _batch = load_and_preprocess_single(_p)
        _hm = compute_gradcam(_batch, _cls_idx)
        _heatmaps.append(resize_heatmap(_hm))
    _avg = np.mean(np.stack(_heatmaps, axis=0), axis=0)
    per_class_avg_heatmap[_cls] = _avg
    per_class_sample_count[_cls] = _n_take
    print(f"  [{_cls}] averaged Grad-CAM over {_n_take} / {_n_available} correctly-classified "
          "VAL samples.")

print()
print("Per-class Grad-CAM sample counts:", per_class_sample_count)


  [angry] averaged Grad-CAM over 100 / 240 correctly-classified VAL samples.
  [disgust] averaged Grad-CAM over 28 / 28 correctly-classified VAL samples.
  [fear] averaged Grad-CAM over 100 / 242 correctly-classified VAL samples.
  [happy] averaged Grad-CAM over 100 / 755 correctly-classified VAL samples.
  [neutral] averaged Grad-CAM over 100 / 359 correctly-classified VAL samples.
  [sad] averaged Grad-CAM over 100 / 330 correctly-classified VAL samples.
  [surprise] averaged Grad-CAM over 100 / 314 correctly-classified VAL samples.

Per-class Grad-CAM sample counts: {'angry': 100, 'disgust': 28, 'fear': 100, 'happy': 100, 'neutral': 100, 'sad': 100, 'surprise': 100}


In [37]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.ravel()
for _i, _cls in enumerate(CLASS_NAMES):
    ax = axes[_i]
    _avg = per_class_avg_heatmap[_cls]
    if _avg is None:
        ax.set_title(f"{_cls} (no samples)")
        ax.axis("off")
        continue
    im = ax.imshow(_avg, cmap="jet")
    ax.set_title(f"{_cls}  (n={per_class_sample_count[_cls]})")
    ax.set_xlabel("x (px)")
    ax.set_ylabel("y (px)")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
for _j in range(len(CLASS_NAMES), len(axes)):
    axes[_j].axis("off")
fig.suptitle("Grad-CAM: class-averaged activation (last conv layer), correctly-classified VAL samples\n"
             "Averaged heatmap only - no individual face image shown")
fig.tight_layout()
_p = os.path.join(NB06_PLOT_DIR, "gradcam_class_average_grid.png")
fig.savefig(_p, dpi=150)
plt.close(fig)
track_write(_p)
print("saved:", _p)


saved: /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/plots/nb06/gradcam_class_average_grid.png


## 11. Grad-CAM — correctly-classified vs incorrectly-classified (pooled across classes)

A seeded sample of correctly-classified and incorrectly-classified VALIDATION predictions
(pooled across all 7 classes) is drawn; Grad-CAM heatmaps (computed against each sample's
**predicted** class) are averaged separately for each group.

In [38]:
N_GRADCAM_SAMPLES_GROUP = 150  # pooled sample size per group (correct / incorrect)

def sample_and_average_gradcam(df_subset, n_take, label):
    n_available = len(df_subset)
    n_take_actual = min(n_take, n_available)
    if n_available == 0:
        print(f"  [{label}] no samples available.")
        return None, 0
    idx = RNG.choice(n_available, size=n_take_actual, replace=False)
    rows = df_subset.iloc[idx]
    heatmaps = []
    for _, row in rows.iterrows():
        batch = load_and_preprocess_single(row["abs_path"])
        hm = compute_gradcam(batch, int(row["predicted_label"]))
        heatmaps.append(resize_heatmap(hm))
    avg = np.mean(np.stack(heatmaps, axis=0), axis=0)
    print(f"  [{label}] averaged Grad-CAM over {n_take_actual} / {n_available} samples.")
    return avg, n_take_actual

_correct_pool = val_result_df[val_result_df["is_correct"]]
_incorrect_pool = val_result_df[~val_result_df["is_correct"]]

avg_heatmap_correct, n_correct_used = sample_and_average_gradcam(
    _correct_pool, N_GRADCAM_SAMPLES_GROUP, "correct")
avg_heatmap_incorrect, n_incorrect_used = sample_and_average_gradcam(
    _incorrect_pool, N_GRADCAM_SAMPLES_GROUP, "incorrect")


  [correct] averaged Grad-CAM over 150 / 2268 samples.
  [incorrect] averaged Grad-CAM over 150 / 1321 samples.


In [39]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, avg, title, n in zip(
    axes, [avg_heatmap_correct, avg_heatmap_incorrect],
    ["Correctly classified", "Incorrectly classified"], [n_correct_used, n_incorrect_used],
):
    if avg is None:
        ax.set_title(f"{title} (no samples)")
        ax.axis("off")
        continue
    im = ax.imshow(avg, cmap="jet")
    ax.set_title(f"{title}  (n={n})")
    ax.set_xlabel("x (px)")
    ax.set_ylabel("y (px)")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.suptitle("Grad-CAM: averaged activation, correct vs incorrect VAL predictions "
             "(pooled across classes)\nAveraged heatmap only - no individual face image shown")
fig.tight_layout()
_p = os.path.join(NB06_PLOT_DIR, "gradcam_correct_vs_incorrect.png")
fig.savefig(_p, dpi=150)
plt.close(fig)
track_write(_p)
print("saved:", _p)


saved: /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/plots/nb06/gradcam_correct_vs_incorrect.png


## 12. Grad-CAM — fear vs sad confusion pair

Three averaged heatmaps: correctly-classified fear, correctly-classified sad, and
fear-samples-misclassified-as-sad. If fewer than ~20 fear-as-sad VALIDATION samples exist,
the actual count is reported and the plot proceeds anyway with a small-sample caveat.

In [40]:
_fear_correct = val_result_df[
    (val_result_df["true_class"] == "fear") & (val_result_df["is_correct"])
]
_sad_correct = val_result_df[
    (val_result_df["true_class"] == "sad") & (val_result_df["is_correct"])
]
_fear_as_sad = val_result_df[
    (val_result_df["true_class"] == "fear") & (val_result_df["predicted_class"] == "sad")
]

print(f"fear correctly classified (VAL): {len(_fear_correct)}")
print(f"sad correctly classified (VAL):  {len(_sad_correct)}")
print(f"fear misclassified as sad (VAL): {len(_fear_as_sad)}")
if 0 < len(_fear_as_sad) < 20:
    print(f"[CAVEAT] only {len(_fear_as_sad)} fear-as-sad VAL samples exist - small-sample "
          "result, proceeding anyway per spec.")
elif len(_fear_as_sad) == 0:
    print("[CAVEAT] zero fear-as-sad VAL samples exist - the corresponding panel will be empty.")

def average_gradcam_for_class(df_subset, class_idx_for_gradcam, n_take, label):
    n_available = len(df_subset)
    n_take_actual = min(n_take, n_available)
    if n_available == 0:
        print(f"  [{label}] no samples available.")
        return None, 0
    idx = RNG.choice(n_available, size=n_take_actual, replace=False)
    paths = df_subset["abs_path"].to_numpy()[idx]
    heatmaps = []
    for p in paths:
        batch = load_and_preprocess_single(p)
        hm = compute_gradcam(batch, class_idx_for_gradcam)
        heatmaps.append(resize_heatmap(hm))
    avg = np.mean(np.stack(heatmaps, axis=0), axis=0)
    print(f"  [{label}] averaged Grad-CAM over {n_take_actual} / {n_available} samples.")
    return avg, n_take_actual

avg_fear_correct, n_fear_correct_used = average_gradcam_for_class(
    _fear_correct, CLASS_TO_IDX["fear"], N_GRADCAM_SAMPLES_PER_CLASS, "fear-correct")
avg_sad_correct, n_sad_correct_used = average_gradcam_for_class(
    _sad_correct, CLASS_TO_IDX["sad"], N_GRADCAM_SAMPLES_PER_CLASS, "sad-correct")
# for fear-as-sad, Grad-CAM is computed against the PREDICTED class (sad) - that is the
# class score whose activation we want to inspect for this confusion pair
avg_fear_as_sad, n_fear_as_sad_used = average_gradcam_for_class(
    _fear_as_sad, CLASS_TO_IDX["sad"], len(_fear_as_sad), "fear-as-sad")


fear correctly classified (VAL): 242
sad correctly classified (VAL):  330
fear misclassified as sad (VAL): 84
  [fear-correct] averaged Grad-CAM over 100 / 242 samples.
  [sad-correct] averaged Grad-CAM over 100 / 330 samples.
  [fear-as-sad] averaged Grad-CAM over 84 / 84 samples.


In [41]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
_panels = [
    (avg_fear_correct, "Fear (correct)", n_fear_correct_used),
    (avg_sad_correct, "Sad (correct)", n_sad_correct_used),
    (avg_fear_as_sad, "Fear misclassified as sad", n_fear_as_sad_used),
]
for ax, (avg, title, n) in zip(axes, _panels):
    if avg is None:
        ax.set_title(f"{title} (no samples)")
        ax.axis("off")
        continue
    im = ax.imshow(avg, cmap="jet")
    ax.set_title(f"{title}  (n={n})")
    ax.set_xlabel("x (px)")
    ax.set_ylabel("y (px)")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.suptitle("Grad-CAM: fear vs sad confusion pair (averaged heatmaps, VAL)\n"
             "Averaged heatmap only - no individual face image shown")
fig.tight_layout()
_p = os.path.join(NB06_PLOT_DIR, "gradcam_fear_vs_sad.png")
fig.savefig(_p, dpi=150)
plt.close(fig)
track_write(_p)
print("saved:", _p)


saved: /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/plots/nb06/gradcam_fear_vs_sad.png


## 13. Review-only plots (identifiable faces) — `review_` prefix, NOT for git

These two plots show identifiable individual face images and are for the supervisor/owner's
own review only. They are prefixed `review_` and must not be committed.

In [42]:
N_REVIEW_OVERLAYS = 10

def overlay_heatmap_on_face(face_img_float, heatmap, alpha=0.4):
    """face_img_float: (H, W, 3) in [0, 255] approx (reconstructed from preprocessed input).
    heatmap: (H, W) in [0, 1]."""
    cmap = plt.get_cmap("jet")
    heatmap_rgb = cmap(heatmap)[..., :3]
    overlay = (1 - alpha) * (face_img_float / 255.0) + alpha * heatmap_rgb
    return np.clip(overlay, 0, 1)

def raw_face_for_display(path):
    """Decode a plain grayscale->RGB 96x96 uint8 image for visualisation (not the
    mobilenet-preprocessed float tensor, so it displays with natural brightness)."""
    raw = tf.io.read_file(path)
    img = tf.io.decode_jpeg(raw, channels=1, dct_method="INTEGER_ACCURATE")
    img = tf.image.resize(img, (NATIVE_SIZE, NATIVE_SIZE), method="bilinear")
    img = tf.cast(tf.round(img), tf.uint8)
    img = tf.image.grayscale_to_rgb(img)
    img = tf.image.resize(img, (INPUT_SIZE, INPUT_SIZE), method="bilinear")
    return img.numpy().astype(np.float32)

_review_pool = val_result_df.sample(
    n=min(N_REVIEW_OVERLAYS, len(val_result_df)), random_state=42
).reset_index(drop=True)

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.ravel()
for _i, row in _review_pool.iterrows():
    ax = axes[_i]
    batch = load_and_preprocess_single(row["abs_path"])
    hm = resize_heatmap(compute_gradcam(batch, int(row["predicted_label"])))
    face = raw_face_for_display(row["abs_path"])
    overlay = overlay_heatmap_on_face(face, hm)
    ax.imshow(overlay)
    _status = "OK" if row["is_correct"] else "WRONG"
    ax.set_title(f"{_status}: true={row['true_class']} pred={row['predicted_class']}", fontsize=9)
    ax.axis("off")
for _j in range(len(_review_pool), len(axes)):
    axes[_j].axis("off")
fig.suptitle("REVIEW ONLY - individual Grad-CAM overlays on real faces (not for git)")
fig.tight_layout()
_p = os.path.join(NB06_PLOT_DIR, "review_gradcam_individual_overlays.png")
fig.savefig(_p, dpi=150)
plt.close(fig)
print("saved (review-only, NOT for git):", _p)


saved (review-only, NOT for git): /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/plots/nb06/review_gradcam_individual_overlays.png


In [43]:
N_REVIEW_MISCLASSIFIED = 12

_misclassified_val = val_result_df[~val_result_df["is_correct"]]
_review_mis = _misclassified_val.sample(
    n=min(N_REVIEW_MISCLASSIFIED, len(_misclassified_val)), random_state=42
).reset_index(drop=True)

_ncols = 4
_nrows = int(np.ceil(len(_review_mis) / _ncols)) if len(_review_mis) else 1
fig, axes = plt.subplots(_nrows, _ncols, figsize=(4 * _ncols, 4 * _nrows))
axes = np.array(axes).reshape(-1)
for _i, row in _review_mis.iterrows():
    ax = axes[_i]
    face = raw_face_for_display(row["abs_path"]) / 255.0
    ax.imshow(face)
    ax.set_title(f"true={row['true_class']}\npred={row['predicted_class']}", fontsize=9)
    ax.axis("off")
for _j in range(len(_review_mis), len(axes)):
    axes[_j].axis("off")
fig.suptitle("REVIEW ONLY - misclassified VAL face thumbnails (not for git)")
fig.tight_layout()
_p = os.path.join(NB06_PLOT_DIR, "review_misclassified_samples.png")
fig.savefig(_p, dpi=150)
plt.close(fig)
print("saved (review-only, NOT for git):", _p)


saved (review-only, NOT for git): /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/plots/nb06/review_misclassified_samples.png


## 14. Export Grad-CAM summary to `../outputs/nb06_gradcam_summary.json`

In [44]:
GRADCAM_SUMMARY = {
    "per_class_sample_counts": per_class_sample_count,
    "correct_vs_incorrect": {"n_correct_used": n_correct_used, "n_incorrect_used": n_incorrect_used},
    "fear_vs_sad": {
        "n_fear_correct_used": n_fear_correct_used,
        "n_sad_correct_used": n_sad_correct_used,
        "n_fear_as_sad_used": n_fear_as_sad_used,
        "small_sample_caveat": bool(n_fear_as_sad_used < 20),
    },
    "calibration": {"ece": ECE, "n_bins": N_CAL_BINS},
    "brightness_confound": {
        "n_misclassified": int(len(misclassified)),
        "fraction_closer_to_predicted_class_brightness": _frac_closer_to_predicted,
        "null_expectation": _null_expectation,
        "delta_vs_null": _confound_delta,
        "method": "compare |intensity - mean_intensity[predicted_class]| vs "
                  "|intensity - mean_intensity[true_class]| for misclassified TEST samples",
        "limitation": "correlational, not causal; small per-class sample sizes among "
                      "misclassifications; brightness is confounded with expression content "
                      "in the source photography",
    },
    "blazeface_confidence_stratification": {
        "n_bins_actual": int(N_BF_BINS_ACTUAL),
        "spearman_rho_confidence_vs_correct": float(_spearman_conf_correct),
        "spearman_p_value": float(_spearman_conf_correct_p),
    },
    "intensity_stratification": {
        "n_bins_actual": int(N_INTENSITY_BINS_ACTUAL),
        "spearman_rho_intensity_vs_correct": float(_spearman_intensity_correct),
        "spearman_p_value": float(_spearman_intensity_correct_p),
    },
    "gradcam_last_conv_layer": {
        "name": GRADCAM_LAST_CONV_LAYER.name,
        "output_shape": list(_rank4_output_shape(GRADCAM_LAST_CONV_LAYER) or []),
        "mode": GRADCAM_MODE,
    },
    "val_accuracy_this_run": _val_acc,
}
p_gc = os.path.join(OUT_DIR, "nb06_gradcam_summary.json")
with open(p_gc, "w", encoding="utf-8") as f:
    json.dump(GRADCAM_SUMMARY, f, indent=2)
track_write(p_gc)
print("saved:", p_gc)


saved: /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/outputs/nb06_gradcam_summary.json


## 15. Demographic limitation — an unresolvable external-validity constraint

**FER-2013 carries no demographic labels.** No demographic attribute (e.g. apparent age,
skin tone, ethnicity) may be inferred from these images anywhere in this project, for two
compounding reasons:

1. **No ground truth to validate against.** There is nothing in FER-2013 to check an
   inferred demographic label against, so any such inference would be unverifiable and
   unfalsifiable within this dataset — it could not be audited for accuracy or bias, which
   makes it worse than useless for a fairness analysis.
2. **Methodological and ethical inappropriateness.** Inferring protected attributes from
   face images without consent or ground-truth labels is not a defensible way to assess
   fairness; a wrong inference could actively fabricate a demographic finding, and even a
   "right" one crosses a line the dataset's collection process never obtained consent for.

Notebook 01 (`docs/ml/01_dataset_exploration_findings.md`) records a qualitative
observation about the composition of the dataset's source images; this notebook does not
restate or re-derive any specific number from that document — it is cited by name for
context only.

**Consequence for this project:** this is treated as an **unresolvable external-validity
constraint** for a Sri Lankan deployment context. No amount of further analysis inside this
notebook, or this dataset, can establish how the model's error patterns vary across the
demographic makeup of an actual Sri Lankan user base, because the labels needed to measure
that do not exist and must not be fabricated. This limitation is reported, not mitigated.


In [45]:
print("Demographic-limitation reminder: FER-2013 has no demographic labels.")
print("No demographic attribute is inferred anywhere in this notebook or project.")
print("See docs/ml/01_dataset_exploration_findings.md for the qualitative dataset-composition note.")
print("This is treated as an unresolvable external-validity constraint for Sri Lankan deployment,")
print("not something this notebook can compute its way out of.")


Demographic-limitation reminder: FER-2013 has no demographic labels.
No demographic attribute is inferred anywhere in this notebook or project.
See docs/ml/01_dataset_exploration_findings.md for the qualitative dataset-composition note.
This is treated as an unresolvable external-validity constraint for Sri Lankan deployment,
not something this notebook can compute its way out of.


## 16. Run metadata — fill `RUN` and save

Final step: the `RUN` dict (redefined in section 0) is filled from the measured values
above and written via `save_run()`, exactly as notebooks 03/04/05 do.

In [46]:
RUN["hyperparameters"] = {
    "input_size": INPUT_SIZE,
    "channels": 3,
    "preprocessing": "grayscale replicated to 3 channels, bilinear resize to 96x96, "
                     "mobilenet_v2.preprocess_input",
    "class_names": CLASS_NAMES,
    "n_gradcam_samples_per_class_requested": N_GRADCAM_SAMPLES_PER_CLASS,
    "n_gradcam_samples_group_requested": N_GRADCAM_SAMPLES_GROUP,
    "n_calibration_bins": N_CAL_BINS,
    "n_blazeface_bins_requested": N_BINS_REQUESTED,
    "n_intensity_bins_requested": N_INTENSITY_BINS_REQUESTED,
    "test_set_touch_count": 0,
    "test_set_note": "test-set results read from nb05_test_probabilities_finetuned.csv only; "
                      "pixel-level brightness stats read raw TEST pixels (no model inference)",
    "val_set_inference_count": 1,
    "threshold_applied": False,
}

RUN["metrics"] = {
    "n_test_predictions_loaded": int(len(test_pred_df)),
    "n_val_used_for_gradcam": int(len(val_df)),
    "test_overall_accuracy_recomputed": _overall_acc,
    "ece": ECE,
    "blazeface_confidence_spearman_rho": float(_spearman_conf_correct),
    "intensity_spearman_rho": float(_spearman_intensity_correct),
    "brightness_confound_fraction_closer_to_predicted": _frac_closer_to_predicted,
    "brightness_confound_delta_vs_null": _confound_delta,
    "gradcam_per_class_sample_counts": per_class_sample_count,
    "gradcam_correct_vs_incorrect_counts": {"correct": n_correct_used, "incorrect": n_incorrect_used},
    "gradcam_fear_vs_sad_counts": {
        "fear_correct": n_fear_correct_used, "sad_correct": n_sad_correct_used,
        "fear_as_sad": n_fear_as_sad_used,
    },
    "val_accuracy_this_run": _val_acc,
}

RUN["notes"] = (
    "Notebook 06 characterises the fine-tuned model's TEST errors by reading "
    "nb05_test_probabilities_finetuned.csv (no re-inference on TEST anywhere in this "
    "notebook), joined to image_flags.csv on basename (join verified exact and complete: "
    f"{len(merged)}/{len(test_pred_df)} rows matched). "
    f"BlazeFace-confidence stratification used {N_BF_BINS_ACTUAL} bins "
    f"(Spearman rho={_spearman_conf_correct:.4f} vs correctness); pixel-intensity "
    f"stratification (read directly from raw TEST pixels, not a model inference) used "
    f"{N_INTENSITY_BINS_ACTUAL} bins (Spearman rho={_spearman_intensity_correct:.4f}). "
    f"Brightness-confound check on {len(misclassified)} misclassified TEST samples: "
    f"{_frac_closer_to_predicted:.4f} fraction closer to the predicted class's brightness "
    f"profile than the true class's (delta vs 0.5 null: {_confound_delta:+.4f}); this is "
    "correlational only, not causal, and confounded with expression content in the source "
    f"photography. Calibration: ECE={ECE:.4f} over {N_CAL_BINS} bins, no threshold selected. "
    "Grad-CAM (the only re-inference in this notebook, run on VALIDATION only, never TEST) "
    f"located the last conv layer at runtime ({GRADCAM_LAST_CONV_LAYER.name!r}), averaged "
    "heatmaps per class (up to 100 correctly-classified VAL samples/class), correct-vs-"
    "incorrect (pooled), and the fear/sad confusion triad "
    f"(fear-as-sad n={n_fear_as_sad_used}, small-sample caveat="
    f"{n_fear_as_sad_used < 20}). Two review_-prefixed plots with identifiable faces were "
    "produced for supervisor review only and are not for git. FER-2013 carries no "
    "demographic labels; no demographic attribute was inferred anywhere in this notebook - "
    "this is documented as an unresolvable external-validity constraint for Sri Lankan "
    "deployment (see docs/ml/01_dataset_exploration_findings.md)."
)

run_path = save_run(RUN)
track_write(run_path)

_written_total = bucket_bytes(_WRITTEN_FILES)
print()
print(f"Total artifact bytes written by this notebook: {_written_total} ({_written_total / 1e6:.2f} MB)")
print()
print("=" * 78)
print("NOTEBOOK 06 COMPLETE")
print("=" * 78)
print(f"TEST accuracy (recomputed): {_overall_acc:.4f}")
print(f"ECE: {ECE:.4f}")
print(f"Brightness-confound delta vs null: {_confound_delta:+.4f}")
print(f"VAL accuracy (this Grad-CAM run): {_val_acc:.4f}")


saved: /mnt/c/Users/Yasindu/Desktop/Chat_Research/IT22638168/ml/fer/outputs/run_20260828_125554.json

Total artifact bytes written by this notebook: 671176 (0.67 MB)

NOTEBOOK 06 COMPLETE
TEST accuracy (recomputed): 0.6289
ECE: 0.3006
Brightness-confound delta vs null: +0.0150
VAL accuracy (this Grad-CAM run): 0.6319
